In [13]:
import pandas as pd
import unicodedata
import re
import numpy as np

from currency_converter import CurrencyConverter
from datetime import date

c = CurrencyConverter()

# -------------------------
# Helpers
# -------------------------
def insert_mapped_column(df, base_col, new_col, mapping):
    if base_col not in df.columns:
        return
    df.insert(df.columns.get_loc(base_col) + 1, new_col, df[base_col].map(mapping))

def clean_text(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = unicodedata.normalize("NFC", s)
    s = s.replace("–", "-").replace("—", "-")
    s = s.replace("’", "'")
    s = re.sub(r"\s+", " ", s)
    return s

def to_lowercase(s):
    if pd.isna(s):
        return s
    return str(s).lower()

def clean_multi_select_to_str(value):
    """Input: 'A;B;C' -> Output: 'a;b;c' (unique+sorted, cleaned).
       Missing -> '' """
    if pd.isna(value):
        return ""
    parts = str(value).split(";")
    cleaned = []
    for p in parts:
        p = clean_text(p)
        if p:
            cleaned.append(p.lower())  # Multi-Select auch direkt lowercasing
    cleaned = sorted(set(cleaned))
    return ";".join(cleaned)

def parse_years(x):
    """StackOverflow YearsCode/YearsCodePro style strings -> float."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s == "":
        return np.nan
    if "less than 1" in s:
        return 0.5
    if "more than 50" in s:
        return 51.0
    try:
        return float(s)
    except:
        return np.nan

def convert_to_usd(currency, comp):
    if pd.isna(currency) or pd.isna(comp):
        return np.nan
    if currency not in c.currencies:
        return np.nan

    # Datumwahl wie bei dir (lassen wir so)
    if currency == "RUB":
        return c.convert(comp, currency, "USD", date=date(2022, 3, 1))
    elif currency == "HRK":
        return c.convert(comp, currency, "USD", date=date(2022, 12, 30))
    else:
        return c.convert(comp, currency, "USD", date=date(2025, 10, 6))

# -------------------------
# 1) Laden
# -------------------------
df = pd.read_csv("../Moritz/survey_results_public.csv")

# -------------------------
# 2) Drop (robust)
# -------------------------
drop_cols = [
    'EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'AILearnHow', 'PurchaseInfluence',
    'ToolCountWork', 'ToolCountPersonal',
    'LanguageAdmired', 'LanguagesHaveEntry', 'LanguagesWantEntry',
    'DatabaseAdmired', 'DatabaseHaveEntry', 'DatabaseWantEntry',
    'PlatformAdmired', 'PlatformHaveEntry', 'PlatformWantEntry',
    'WebframeAdmired', 'WebframeHaveEntry', 'WebframeWantEntry',
    'DevEnvsAdmired', 'DevEnvHaveEntry', 'DevEnvWantEntry',
    'OpSysPersonal use', 'OpSysProfessional use',
    'OfficeStackAsyncAdmired', 'OfficeStackHaveEntry', 'OfficeStackWantEntry',

    # ⚠️ Diese beiden würde ich für Profile eher behalten – wenn ihr sie wollt, einfach rausnehmen:
    'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith',

    'CommPlatformAdmired', 'CommPlatformHaveEntr', 'CommPlatformWantEntr',
    'AIModelsAdmired', 'AIModelsHaveEntry', 'AIModelsWantEntry',
    'AISent', 'AIAcc', 'AIComplex',
    'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task",
    'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI',
    'AIToolCurrently mostly AI', 'AIFrustration', 'AIExplain',
    'AIAgentChange', 'AgentUsesGeneral',
    'AIAgentImpactSomewhat agree', 'AIAgentImpactNeutral',
    'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree',
    'AIAgentImpactStrongly disagree',
    'AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat disagree',
    'AIAgentChallengesStrongly agree', 'AIAgentChallengesSomewhat agree',
    'AIAgentChallengesStrongly disagree',
    'AIAgentKnowledge', 'AIAgentKnowWrite',
    'AIAgentOrchestration', 'AIAgentOrchWrite',
    'AIAgentObserveSecure', 'AIAgentObsWrite',
    'AIAgentExternal', 'AIAgentExtWrite',
    'AIHuman', 'AIOpen'
]
df = df.drop(columns=drop_cols, errors="ignore")

prefixes = ("TechEndorse", "TechOppose", "JobSatPoints", "SO")
df = df.drop(columns=df.columns[df.columns.str.startswith(prefixes)], errors="ignore")

# -------------------------
# 3) EdLevel vereinfachen
# -------------------------
if "EdLevel" in df.columns:
    df["EdLevel"] = df["EdLevel"].astype(str).str.split("(").str[0].str.strip()

# -------------------------
# 4) RemoteWork/ Age Mappings
# -------------------------
remote_map = {
    "Remote": 0,
    "In-person": 1,
    "Hybrid (some remote, leans heavy to in-person)": 0.75,
    "Hybrid (some in-person, leans heavy to flexibility)": 0.25,
    "Your choice (very flexible, you can come in when you want or just as needed)": 0.5
}
insert_mapped_column(df, "RemoteWork", "RemoteCategoryNum", remote_map)
df["RemoteMissing"] = df["RemoteCategoryNum"].isna().astype("int8")
df["RemoteCategoryNum"] = df["RemoteCategoryNum"].fillna(0.5)
age_map = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70
}
insert_mapped_column(df, "Age", "AgeNum", age_map)

age_map2 = {
    "Under 18 years old": 18,
    "18-24 years old": 24,
    "25-34 years old": 34,
    "35-44 years old": 44,
    "45-54 years old": 54,
    "55-64 years old": 64,
    "65 years or older": 100
}
insert_mapped_column(df, "Age", "MaxAge", age_map2)

# Entferne fehlende AgeNum + >65
if "AgeNum" in df.columns:
    df = df[df["AgeNum"].notna()]
    df = df[df["AgeNum"] <= 65]

# -------------------------
# 5) YearsCode/WorkExp numeric machen (wichtig für eure Plausi-Filter!)
# -------------------------
if "YearsCode" in df.columns:
    df["YearsCode"] = df["YearsCode"].apply(parse_years)

# WorkExp ist oft schon numerisch, aber sicher ist sicher
if "WorkExp" in df.columns:
    df["WorkExp"] = pd.to_numeric(df["WorkExp"], errors="coerce")

# -------------------------
# 6) Object-Spalten lowercasing (Country/Currency ausnehmen)
# -------------------------
exclude_columns = {"Country", "Currency"}
for col in df.select_dtypes(include=["object"]).columns:
    if col not in exclude_columns:
        df[col] = df[col].apply(to_lowercase)

# -------------------------
# 7) Multi-Select cleanen + Count-Spalten erzeugen
# -------------------------
multi_select_cols = [
    'LanguageHaveWorkedWith', 'LanguageWantToWorkWith',
    'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith',
    'PlatformHaveWorkedWith', 'PlatformWantToWorkWith',
    'WebframeHaveWorkedWith', 'WebframeWantToWorkWith',
    'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith',
    'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith',
    'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith',
    'AIAgent_Uses'
]
multi_select_cols = [c for c in multi_select_cols if c in df.columns]

for col in multi_select_cols:
    df[col] = df[col].apply(clean_multi_select_to_str)
    # df[f"{col}_Count"] = df[col].apply(lambda s: 0 if s == "" else len(s.split(";"))).astype("int16")

# -------------------------
# 8) Plausibilitätsfilter (nur wenn Spalten da sind)
# -------------------------
if {"WorkExp", "MaxAge"}.issubset(df.columns):
    df = df[~(df["WorkExp"] > (df["MaxAge"] - 16))]

if {"YearsCode", "MaxAge"}.issubset(df.columns):
    df = df[~(df["YearsCode"] > (df["MaxAge"] - 6))]  # (Start coding mit ~6 erlaubt)

# -------------------------
# 9) Salary in USD (ConvertedCompTotal)
# -------------------------
# Currency auf 3 Buchstaben
if "Currency" in df.columns:
    df["Currency"] = df["Currency"].astype(str).str[:3]
    df.loc[df["Currency"].isin(["nan", "None"]), "Currency"] = np.nan

# CompTotal numeric erzwingen
if "CompTotal" in df.columns:
    df["CompTotal"] = pd.to_numeric(df["CompTotal"], errors="coerce")

if {"Currency", "CompTotal"}.issubset(df.columns):
    df["ConvertedCompTotal"] = df.apply(lambda row: convert_to_usd(row["Currency"], row["CompTotal"]), axis=1)

drop_after_salary = ["CompTotal", "Currency", "ConvertedCompYearly"]
drop_after_salary = [c for c in drop_after_salary if c in df.columns]
df = df.drop(columns=drop_after_salary)
keep_cols = ["ConvertedCompTotal", "WorkExp", "YearsCode"]
keep_cols = [c for c in keep_cols if c in df.columns]

for col in keep_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    q95 = df[col].quantile(0.95)
    before = len(df)
    df = df[df[col].isna() | (df[col] <= q95)]
    print(f"{col}: kept <= q95={q95:.4g} | removed {before-len(df)} rows")

# -------------------------
# 10) Speichern
# -------------------------
df.to_csv("survey_results_cleaned.csv", index=False)
print("Saved:", df.shape, "-> survey_results_cleaned.csv")


ConvertedCompTotal: kept <= q95=2.35e+05 | removed 1102 rows
WorkExp: kept <= q95=30 | removed 1931 rows
YearsCode: kept <= q95=34 | removed 1855 rows
Saved: (42626, 45) -> survey_results_cleaned.csv


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42626 entries, 0 to 49122
Data columns (total 44 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ResponseId                      42626 non-null  int64  
 1   MainBranch                      42626 non-null  object 
 2   Age                             42626 non-null  object 
 3   MaxAge                          42626 non-null  float64
 4   AgeNum                          42626 non-null  float64
 5   EdLevel                         42626 non-null  object 
 6   Employment                      41818 non-null  object 
 7   WorkExp                         36621 non-null  float64
 8   LearnCodeAI                     38829 non-null  object 
 9   YearsCode                       36788 non-null  float64
 10  DevType                         37379 non-null  object 
 11  OrgSize                         29646 non-null  object 
 12  ICorPM                          28841

In [3]:
df = pd.read_csv("survey_results_cleaned.csv")

In [4]:
df

,ResponseId,MainBranch,Age,MaxAge,AgeNum,EdLevel,Employment,WorkExp,LearnCodeAI,YearsCode,...,OfficeStackAsyncWantToWorkWith,AIModelsChoice,AIModelsHaveWorkedWith,AIModelsWantToWorkWith,AISelect,AIAgents,AIAgent_Uses,ConvertedCompYearly,JobSat,ConvertedCompTotal
0,1,i am a developer by profession,25-34 years old,34.0,29.0,master’s degree,employed,8.0,"yes, i learned how to use ai-enabled tools for...",14.0,...,markdown file,yes,openai gpt (chatbot models);openai image gener...,NaN,"yes, i use ai tools monthly or infrequently","yes, i use ai agents at work monthly or infreq...",software engineering,61256.0,10.0,61659.84
1,2,i am a developer by profession,25-34 years old,34.0,29.0,associate degree,employed,2.0,"yes, i learned how to use ai-enabled tools for...",10.0,...,confluence;github;jira,yes,openai gpt (chatbot models),openai gpt (chatbot models),"yes, i use ai tools weekly","no, and i don't plan to",NaN,104413.0,9.0,105102.00
2,3,i am a developer by profession,35-44 years old,44.0,39.0,bachelor’s degree,"independent contractor, freelancer, or self-em...",10.0,"yes, i learned how to use ai-enabled tools for...",12.0,...,github;gitlab;jira,yes,gemini (flash general purpose models);openai g...,gemini (flash general purpose models);gemini (...,"yes, i use ai tools daily","yes, i use ai agents at work weekly",software engineering,53061.0,8.0,NaN
3,4,i am a developer by profession,35-44 years old,44.0,39.0,bachelor’s degree,employed,4.0,"yes, i learned how to use ai-enabled tools for...",5.0,...,gitlab;jira,no,NaN,NaN,"yes, i use ai tools weekly","yes, i use ai agents at work monthly or infreq...",software engineering,36197.0,6.0,36435.36
4,5,i am a developer by profession,35-44 years old,44.0,39.0,master’s degree,"independent contractor, freelancer, or self-em...",21.0,"yes, i learned how to use ai-enabled tools for...",22.0,...,azure devops;github;jira,yes,openai gpt (chatbot models),openai gpt (chatbot models),"yes, i use ai tools weekly","no, and i don't plan to",NaN,60000.0,7.0,60000.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47509,49119,i am a developer by profession,25-34 years old,34.0,29.0,bachelor’s degree,employed,9.0,"yes, i learned how to use ai-enabled tools req...",13.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN
47510,49120,i am a developer by profession,35-44 years old,44.0,39.0,bachelor’s degree,employed,13.0,"yes, i learned how to use ai-enabled tools req...",15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47511,49121,i am a developer by profession,25-34 years old,34.0,29.0,secondary school,employed,2.0,"no, i didn't spend time learning in the past year",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47512,49122,i am a developer by profession,25-34 years old,34.0,29.0,associate degree,employed,10.0,"yes, i learned how to use ai-enabled tools for...",14.0,...,asana;github;obsidian,no,NaN,NaN,"yes, i use ai tools daily","no, i use ai exclusively in copilot/autocomple...",marketing,NaN,7.0,58390.00


In [2]:
import pandas as pd
import unicodedata
import re
import numpy as np

from currency_converter import CurrencyConverter

c = CurrencyConverter()
from datetime import date

# df = pd.read_csv("survey_results_public.csv")
# df.shape


In [3]:
drop_cols = ['EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'AILearnHow', 'PurchaseInfluence',
             'ToolCountWork', 'ToolCountPersonal', 'LanguageAdmired', 'LanguagesHaveEntry',
             'LanguagesWantEntry', 'DatabaseAdmired', 'DatabaseHaveEntry', 'DatabaseWantEntry',
             'PlatformAdmired', 'PlatformHaveEntry', 'PlatformWantEntry', 'WebframeAdmired',
             'WebframeHaveEntry', 'WebframeWantEntry', 'DevEnvsAdmired', 'DevEnvHaveEntry',
             'DevEnvWantEntry', 'OpSysPersonal use', 'OpSysProfessional use',
             'OfficeStackAsyncAdmired', 'OfficeStackHaveEntry', 'OfficeStackWantEntry',
             'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'CommPlatformAdmired',
             'CommPlatformHaveEntr', 'CommPlatformWantEntr', 'AIModelsAdmired',
             'AIModelsHaveEntry', 'AIModelsWantEntry', 'AISent', 'AIAcc', 'AIComplex',
             'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task",
             'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI',
             'AIToolCurrently mostly AI', 'AIFrustration', 'AIExplain', 'AIAgentChange',
             'AgentUsesGeneral', 'AIAgentImpactSomewhat agree', 'AIAgentImpactNeutral',
             'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree',
             'AIAgentImpactStrongly disagree', 'AIAgentChallengesNeutral',
             'AIAgentChallengesSomewhat disagree', 'AIAgentChallengesStrongly agree',
             'AIAgentChallengesSomewhat agree', 'AIAgentChallengesStrongly disagree',
             'AIAgentKnowledge', 'AIAgentKnowWrite', 'AIAgentOrchestration', 'AIAgentOrchWrite',
             'AIAgentObserveSecure', 'AIAgentObsWrite', 'AIAgentExternal', 'AIAgentExtWrite',
             'AIHuman', 'AIOpen']

df = df.drop(columns=drop_cols, errors="ignore")

prefixes = ("TechEndorse", "TechOppose", "JobSatPoints", "SO")
df = df.drop(columns=df.columns[df.columns.str.startswith(prefixes)], errors="ignore")

df.shape

(49123, 43)

In [14]:
def insert_mapped_column(df, base_col, new_col, mapping):
    df.insert(df.columns.get_loc(base_col) + 1, new_col, df[base_col].map(mapping))

def clean_text(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = unicodedata.normalize("NFC", s)
    s = s.replace("–", "-").replace("—", "-")
    s = s.replace("’", "'")
    s = re.sub(r"\s+", " ", s)
    return s

def to_lowercase(s):
    if pd.isna(s):
        return s
    return str(s).lower()

def clean_multi_select(value):
    if pd.isna(value):
        return []
    splitted = str(value).split(";")
    cleaned = []
    for p in splitted:
        p = clean_text(p)
        if p:
            cleaned.append(p)
    cleaned = list(set(cleaned))
    cleaned.sort()
    return cleaned

In [5]:
if "EdLevel" in df.columns:
    df["EdLevel"] = df["EdLevel"].astype(str).str.split("(").str[0].str.strip()

In [6]:
remote_map = {
    "Remote": 0,
    "In-person": 1,
    "Hybrid (some remote, leans heavy to in-person)": 0.75,
    "Hybrid (some in-person, leans heavy to flexibility)": 0.25,
    "Your choice (very flexible, you can come in when you want or just as needed)": 0.5
}
age_map = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70
}
age_map2 = {
    "Under 18 years old": 18,
    "18-24 years old": 24,
    "25-34 years old": 34,
    "35-44 years old": 44,
    "45-54 years old": 54,
    "55-64 years old": 64,
    "65 years or older": 100
}

if "RemoteWork" in df.columns:
    insert_mapped_column(df, "RemoteWork", "RemoteCategoryNum", remote_map)

if "Age" in df.columns:
    insert_mapped_column(df, "Age", "AgeNum", age_map)
    insert_mapped_column(df, "Age", "MaxAge", age_map2)

df[["RemoteWork", "RemoteCategoryNum", "Age", "AgeNum", "MaxAge"]].head()


,RemoteWork,RemoteCategoryNum,Age,AgeNum,MaxAge
0,Remote,0.00,25-34 years old,29.0,34.0
1,"Hybrid (some in-person, leans heavy to flexibi...",0.25,25-34 years old,29.0,34.0
2,NaN,NaN,35-44 years old,39.0,44.0
3,Remote,0.00,35-44 years old,39.0,44.0
4,NaN,NaN,35-44 years old,39.0,44.0


In [7]:
# Über 65 entfernen (und sofort eine echte Kopie erzeugen)
df = df[df["AgeNum"].isna() | (df["AgeNum"] <= 65)].copy()

# Numeric machen (sicher, ohne SettingWithCopyWarning)
for col in ["WorkExp", "YearsCode", "CompTotal"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


In [8]:
multi_select_cols = [
    'LanguageHaveWorkedWith', 'LanguageWantToWorkWith',
    'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith',
    'PlatformHaveWorkedWith', 'PlatformWantToWorkWith',
    'WebframeHaveWorkedWith', 'WebframeWantToWorkWith',
    'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith',
    'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith',
    'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith',
    'AIAgent_Uses'
]

exclude_columns = ["Country", "Currency"]
category_columns = df.select_dtypes(include=["object"]).columns.tolist()

for col in category_columns:
    if col not in exclude_columns:
        df[col] = df[col].apply(to_lowercase)

for col in multi_select_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_multi_select)


In [9]:
# WorkExp <= MaxAge - 16
mask_we = df["WorkExp"].notna() & df["MaxAge"].notna()
df = df[~(mask_we & (df["WorkExp"] > (df["MaxAge"] - 16)))]

# YearsCode <= MaxAge - 6
mask_yc = df["YearsCode"].notna() & df["MaxAge"].notna()
df = df[~(mask_yc & (df["YearsCode"] > (df["MaxAge"] - 6)))]

df.shape


(47892, 46)

In [22]:
if "Currency" in df.columns:
    df["Currency"] = df["Currency"].astype(str).str[:3]


In [23]:
def convert_to_usd(currency, comp):
    if pd.isna(currency) or pd.isna(comp):
        return np.nan
    if currency not in c.currencies:
        return np.nan
    if currency == "RUB":
        return c.convert(comp, currency, 'USD', date=date(2022, 3, 1))
    elif currency == "HRK":
        return c.convert(comp, currency, 'USD', date=date(2022, 12, 30))
    else:
        return c.convert(comp, currency, 'USD', date=date(2025, 10, 6))


df["ConvertedComp_USD"] = np.nan

mask_valid = df["Currency"].notna() & df["CompTotal"].notna() & (df["CompTotal"] > 0)
for curr in sorted(df.loc[mask_valid, "Currency"].unique()):
    m = mask_valid & (df["Currency"] == curr)
    df.loc[m, "ConvertedComp_USD"] = df.loc[m, "CompTotal"].apply(lambda x: convert_to_usd(curr, x))

df[["Currency", "CompTotal", "ConvertedComp_USD"]].head(10)


,Currency,CompTotal,ConvertedComp_USD
0,EUR,52800.0,61659.84
1,EUR,90000.0,105102.00
2,UAH,2214000.0,NaN
3,EUR,31200.0,36435.36
4,USD,60000.0,60000.00
5,USD,120000.0,120000.00
6,USD,6240.0,6240.00
7,USD,72000.0,72000.00
8,USD,70000.0,70000.00
9,USD,5400.0,5400.00


In [24]:
df

,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat,ConvertedCompTotal,ConvertedComp_USD
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0,NaN,61659.84
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0,NaN,105102.00
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",None of the above,10.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,"Understand how things actually work, problem s...",53061.0,8.0,NaN,NaN
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,4.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to...,NaN,36197.0,6.0,NaN,36435.36
4,5,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...","Caring for dependents (children, elderly, etc.)",21.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,"critical thinking, the skill to define the tas...",60000.0,7.0,NaN,60000.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49118,49119,I am a developer by profession,25-34 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,NaN,9.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools req...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN,NaN
49119,49120,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,"Caring for dependents (children, elderly, etc.)",13.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools req...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49120,49121,I am a developer by profession,25-34 years old,"Secondary school (e.g. American high school, G...",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","No, I didn't spend time learning in the past year",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49121,49122,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,None of the above;Engaged in paid work (20-29 ...,10.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,NaN,58390.00


In [25]:
df.to_csv("survey_results_cleaned.csv", index=False)
df.shape


(49123, 172)

In [ ]:
import pandas as pd
import unicodedata
import re

import numpy as np

from currency_converter import CurrencyConverter
c = CurrencyConverter()
from datetime import date

# csv als DataFrame einlesen
df = pd.read_csv("../Moritz/survey_results_public.csv")

# bestimmte Spalten entfernen, die weiter nicht gebraucht werden
drop_cols = ['EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'AILearnHow', 'PurchaseInfluence', 'ToolCountWork', 'ToolCountPersonal', 'LanguageAdmired', 'LanguagesHaveEntry', 'LanguagesWantEntry', 'DatabaseAdmired', 'DatabaseHaveEntry', 'DatabaseWantEntry', 'PlatformAdmired', 'PlatformHaveEntry', 'PlatformWantEntry', 'WebframeAdmired', 'WebframeHaveEntry', 'WebframeWantEntry', 'DevEnvsAdmired', 'DevEnvHaveEntry', 'DevEnvWantEntry', 'OpSysPersonal use', 'OpSysProfessional use', 'OfficeStackAsyncAdmired', 'OfficeStackHaveEntry', 'OfficeStackWantEntry', 'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'CommPlatformAdmired', 'CommPlatformHaveEntr', 'CommPlatformWantEntr', 'AIModelsAdmired', 'AIModelsHaveEntry', 'AIModelsWantEntry', 'AISent', 'AIAcc', 'AIComplex', 'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task", 'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI', 'AIToolCurrently mostly AI', 'AIFrustration', 'AIExplain', 'AIAgentChange', 'AgentUsesGeneral', 'AIAgentImpactSomewhat agree', 'AIAgentImpactNeutral', 'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree', 'AIAgentImpactStrongly disagree', 'AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat disagree', 'AIAgentChallengesStrongly agree', 'AIAgentChallengesSomewhat agree', 'AIAgentChallengesStrongly disagree', 'AIAgentKnowledge', 'AIAgentKnowWrite', 'AIAgentOrchestration', 'AIAgentOrchWrite', 'AIAgentObserveSecure', 'AIAgentObsWrite', 'AIAgentExternal', 'AIAgentExtWrite', 'AIHuman', 'AIOpen']

df = df.drop(columns=drop_cols)
prefixes = ("TechEndorse", "TechOppose", "JobSatPoints", "SO")
df = df.drop(columns=df.columns[df.columns.str.startswith(prefixes)])

# insert Mappings in DatenFrame
def insert_mapped_column(df, base_col, new_col, mapping):
    df.insert(
        df.columns.get_loc(base_col) + 1,
        new_col,
        df[base_col].map(mapping)
    )

# Text vereinheitlichen und Stolperfallen eliminieren
def clean_text(s):
    if pd.isna(s):
        return s
    s = str(s).strip() # Leerzeichen am Anfang und Ende weg
    s = unicodedata.normalize("NFC", s) # Unicode normalisieren
    s = s.replace("–", "-").replace("—", "-") # Bindestriche / Spiegelstriche vereinheitlichen
    s = s.replace("’", "'") # Apostrophen vereinheitlichen
    s = re.sub(r"\s+", " ", s)
    return s

# Zeileninhalt -> lowercase
def to_lowercase(s):
    if pd.isna(s):
        return s
    return s.lower()

# Mulit-Select Spalten cleanen
def clean_multi_select(value):
    if pd.isna(value):
        return []

    splitted = str(value).split(";")

    cleaned = []
    for p in splitted:
        p = clean_text(p)
        if p:
            cleaned.append(p)
    cleaned = list(set(cleaned))
    cleaned.sort()
    return cleaned

#EdLevel splitten um nur EdLevel anzuzeigen
df["EdLevel"] = df["EdLevel"].str.split("(").str[0].str.strip()

#RemoteWork auf numerische Werte mappen
remote_map = {
    "Remote": 0,
    "In-person": 1,
    "Hybrid (some remote, leans heavy to in-person)": 0.75,
    "Hybrid (some in-person, leans heavy to flexibility)": 0.25,
    "Your choice (very flexible, you can come in when you want or just as needed)": 0.5
}
insert_mapped_column(df, "RemoteWork", "RemoteCategoryNum", remote_map)

# Age zu numerischen Werten mappen, immer Mittelwert der ranges
age_map = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70
}
insert_mapped_column(df, "Age", "AgeNum", age_map)

#Age zu numerischen Werten mappen, immer größter Wert der Spalte
age_map2 = {
    "Under 18 years old": 18,
    "18-24 years old": 24,
    "25-34 years old": 34,
    "35-44 years old": 44,
    "45-54 years old": 54,
    "55-64 years old": 64,
    "65 years or older": 100
}
insert_mapped_column(df, "Age", "MaxAge", age_map2)


#Über 65-jährige entfernen
df = df[df['AgeNum'] <= 65]

multi_select_cols = [
    'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsHaveWorkedWith',     'AIModelsWantToWorkWith', 'AIAgent_Uses'
]

exclude_columns = ["Country", "Currency"]
category_columns = df.select_dtypes(include=["object"]).columns.tolist()

for col in category_columns:
    if col not in exclude_columns:
        df[col] = df[col].apply(to_lowercase)

for col in multi_select_cols:
    df[col] = df[col].apply(clean_multi_select)

# Keep only rows where WorkExp is NOT greater than (MaxAge - 16)
df = df[~(df['WorkExp'] > (df['MaxAge'] - 16))]

# Keep only rows where YearsCode is NOT greater than (MaxAge - 16)
df = df[~(df['YearsCode'] > (df['MaxAge'] - 6))]


# Mit Currency Converter Jahresgehalt in USD umwandeln

# Currency Spalte alles nach den ersten 3 Buchstaben abschneiden
df['Currency'] = df['Currency'].str[:3]

# Spalte CompTotal in USD umwandeln und in Spalte convertedCompTotal speichern

def convert_to_usd(currency, comp):
    if currency not in c.currencies:
        return np.nan
    if currency == "RUB":
        converted = c.convert(comp, currency, 'USD', date=date(2022, 3, 1))
    elif currency == "HRK":
        converted = c.convert(comp, currency, 'USD', date=date(2022, 12, 30))
    else:
        converted = c.convert(comp, currency, 'USD', date=date(2025, 10, 6))
    return converted

df['ConvertedCompTotal'] = df.apply(lambda row: convert_to_usd(row['Currency'], row['CompTotal']), axis=1)




df.to_csv("survey_results_cleaned.csv", index=False)

In [ ]:
df

# 🧹 Datenbereinigung & Vereinheitlichung – To-Do Liste

Diese To-Do-Liste beschreibt alle notwendigen Schritte, um die Survey-CSV-Datei zu bereinigen, zu vereinheitlichen und für spätere Analysen oder Visualisierungen nutzbar zu machen.

---

## 1. Fehlende Werte standardisieren

### Kategoriale Spalten
- `NaN` → **"Keine Angabe"**
- Datentyp `object` beibehalten

### Numerische Spalten
- `NaN` **nicht ersetzen**
- Datentyp `int`/`float` belassen
  → wichtig für statistische Auswertungen (Durchschnitt, Median, Histogramme)

In [5]:
# Alle Spalten mit numerischen Werten in einen DataFrame packen
numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Alle Spalten mit nicht numerischen Werten in einen DataFrame packen
category_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Anzahl numerischer Spalten:", len(numeric_columns)) #Anzahl: 42
print("Anzahl kategorischer Spalten:", len(category_columns)) #Anzahl: 106

print("\nBeispiele numerischer Spalten:", numeric_columns[:10])
print("\nBeispiele kategorischer Spalten:", category_columns[:10])

Anzahl numerischer Spalten: 43
Anzahl kategorischer Spalten: 106

Beispiele numerischer Spalten: ['MaxAge', 'AgeNum', 'WorkExp', 'YearsCode', 'RemoteCategoryNum', 'TechEndorse_1', 'TechEndorse_2', 'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5']

Beispiele kategorischer Spalten: ['MainBranch', 'Age', 'EdLevel', 'Employment', 'EmploymentAddl', 'LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow', 'DevType']


## 2. Kategorische Text-Spalten bereinigen

### Ziel
Eine einheitliche und konsistente Darstellung, um spätere Gruppierungen und Analysen zu erleichtern.

### Maßnahmen
- Whitespace entfernen (Trimmen)
- Einheitliche Groß-/Kleinschreibung (z. B. `title()` oder `lower()`)
- Zusammenführen identischer kategorischer Werte mit unterschiedlicher Schreibweise
  *Beispiel: „Self taught“ und „self-taught“*
- Optional: Seltene Kategorien in **"Other"** gruppieren

### Beispiele betroffener Spalten
- `MainBranch`
- `EdLevel`
- `Employment`
- `Country`
- `OrgSize`
- `Industry`
- `AISelect`
- `AIPrimaryUse`

In [9]:
# Currency außen vor wegen den Währungscodes wie USD oder EUR
exclude_columns = ["Country", "Currency"]

for col in category_columns:
    if col not in exclude_columns:
        df[col] = df[col].apply(to_lowercase)

## 3. Mehrfachauswahl-Spalten vereinheitlichen (`;`-getrennte Werte)

### Typische Probleme
- Uneinheitliche Formatierungen
- Semikolon-separierte Werte
- NaN-Werte
- Inkonsistente Reihenfolgen

### Maßnahmen
- Aufsplitten in Listen
  `"Python; JavaScript"` → `["Python", "JavaScript"]`
- Werte trimmen
- Duplikate in Listen entfernen
- Optionale alphabetische Sortierung der Werte
- NaN → **leere Liste** oder **"Keine Angabe"**

### Beispiele
- `DevType`
- `LanguageHaveWorkedWith`
- `LanguageWantToWorkWith`
- `DatabaseHaveWorkedWith`
- `ToolsTechHaveWorkedWith`
- `PlatformHaveWorkedWith`

In [10]:
# Ausgabe aller Spalten, die Semikolon-getrennt sind
[col for col in df.columns if df[col].astype(str).str.contains(";").any()]

['EmploymentAddl',
 'LearnCode',
 'AILearnHow',
 'TechEndorse_13_TEXT',
 'TechOppose_15_TEXT',
 'JobSatPoints_15_TEXT',
 'LanguageHaveWorkedWith',
 'LanguageWantToWorkWith',
 'LanguageAdmired',
 'LanguagesHaveEntry',
 'LanguagesWantEntry',
 'DatabaseHaveWorkedWith',
 'DatabaseWantToWorkWith',
 'DatabaseAdmired',
 'DatabaseHaveEntry',
 'DatabaseWantEntry',
 'PlatformHaveWorkedWith',
 'PlatformWantToWorkWith',
 'PlatformAdmired',
 'PlatformWantEntry',
 'WebframeHaveWorkedWith',
 'WebframeWantToWorkWith',
 'WebframeAdmired',
 'WebframeHaveEntry',
 'WebframeWantEntry',
 'DevEnvsHaveWorkedWith',
 'DevEnvsWantToWorkWith',
 'DevEnvsAdmired',
 'DevEnvHaveEntry',
 'DevEnvWantEntry',
 'OpSysPersonal use',
 'OpSysProfessional use',
 'OfficeStackAsyncHaveWorkedWith',
 'OfficeStackAsyncWantToWorkWith',
 'OfficeStackAsyncAdmired',
 'OfficeStackHaveEntry',
 'CommPlatformHaveWorkedWith',
 'CommPlatformWantToWorkWith',
 'CommPlatformAdmired',
 'CommPlatformHaveEntr',
 'CommPlatformWantEntr',
 'AIModels

In [11]:
# Alle Semikolon-getrennten Spalten in einer Liste
multi_select_cols = [
 'EmploymentAddl','LearnCode','AILearnHow','TechEndorse_13_TEXT','TechOppose_15_TEXT',
 'JobSatPoints_15_TEXT','LanguageHaveWorkedWith','LanguageWantToWorkWith','LanguageAdmired',
 'LanguagesHaveEntry','LanguagesWantEntry','DatabaseHaveWorkedWith','DatabaseWantToWorkWith',
 'DatabaseAdmired','DatabaseHaveEntry','DatabaseWantEntry','PlatformHaveWorkedWith',
 'PlatformWantToWorkWith','PlatformAdmired','PlatformWantEntry','WebframeHaveWorkedWith',
 'WebframeWantToWorkWith','WebframeAdmired','WebframeHaveEntry','WebframeWantEntry',
 'DevEnvsHaveWorkedWith','DevEnvsWantToWorkWith','DevEnvsAdmired','DevEnvHaveEntry',
 'DevEnvWantEntry','OpSysPersonal use','OpSysProfessional use',
 'OfficeStackAsyncHaveWorkedWith','OfficeStackAsyncWantToWorkWith','OfficeStackAsyncAdmired',
 'OfficeStackHaveEntry','CommPlatformHaveWorkedWith','CommPlatformWantToWorkWith',
 'CommPlatformAdmired','CommPlatformHaveEntr','CommPlatformWantEntr',
 'AIModelsHaveWorkedWith','AIModelsWantToWorkWith','AIModelsAdmired',
 'AIToolCurrently partially AI',"AIToolDon't plan to use AI for this task",
 'AIToolPlan to partially use AI','AIToolPlan to mostly use AI','AIToolCurrently mostly AI',
 'AIFrustration','AIExplain','AIAgent_Uses','AgentUsesGeneral',
 'AIAgentImpactSomewhat agree','AIAgentImpactNeutral','AIAgentImpactSomewhat disagree',
 'AIAgentImpactStrongly agree','AIAgentImpactStrongly disagree',
 'AIAgentChallengesNeutral','AIAgentChallengesSomewhat disagree',
 'AIAgentChallengesStrongly agree','AIAgentChallengesSomewhat agree',
 'AIAgentChallengesStrongly disagree','AIAgentKnowledge','AIAgentKnowWrite',
 'AIAgentOrchestration','AIAgentOrchWrite','AIAgentObserveSecure','AIAgentObsWrite',
 'AIAgentExternal','AIAgentExtWrite','AIHuman','AIOpen'
]

In [12]:
def clean_multi_select(value):
    if pd.isna(value):
        return []

    splitted = str(value).split(";")

    cleaned = []
    for p in splitted:
        clean_text(p)
        if p:
            cleaned.append(p)

    cleaned = list(set(cleaned))

    cleaned.sort()

    return cleaned

In [13]:
for col in multi_select_cols:
    df[col] = df[col].apply(clean_multi_select)

In [14]:
df['YearsCode']

ResponseId
1        14.0
2        10.0
3        12.0
4         5.0
5        22.0
         ... 
49119    13.0
49120    15.0
49121     NaN
49122    14.0
49123    15.0
Name: YearsCode, Length: 47803, dtype: float64

In [15]:
df.to_csv("survey_results_shortened.csv", index=False)


## Filter die Zeilen raus, die mehr Berufs-Erfahrung haben als Lebensalter - 16 Jahre

In [24]:
# Keep only rows where WorkExp is NOT greater than (MaxAge - 16)
df = df[~(df['WorkExp'] > (df['MaxAge'] - 16))]

# Display the first few rows of the filtered dataframe
df.head()

,MainBranch,Age,MaxAge,AgeNum,EdLevel,Employment,WorkExp,LearnCodeAI,YearsCode,DevType,...,OfficeStackAsyncHaveWorkedWith,OfficeStackAsyncWantToWorkWith,AIModelsChoice,AIModelsHaveWorkedWith,AIModelsWantToWorkWith,AISelect,AIAgents,AIAgent_Uses,ConvertedCompYearly,JobSat
ResponseId,,,,,,,,,,,,,,,,,,,,,
1,I am a developer by profession,25-34 years old,34.0,29.0,Master’s degree,Employed,8.0,"Yes, I learned how to use AI-enabled tools for...",14.0,"Developer, mobile",...,Confluence;GitHub;GitLab;Jira;Markdown File,Markdown File,Yes,openAI GPT (chatbot models);openAI Image gener...,NaN,"Yes, I use AI tools monthly or infrequently","Yes, I use AI agents at work monthly or infreq...",Software engineering,61256.0,10.0
2,I am a developer by profession,25-34 years old,34.0,29.0,Associate degree,Employed,2.0,"Yes, I learned how to use AI-enabled tools for...",10.0,"Developer, back-end",...,Confluence;GitHub;Jira,Confluence;GitHub;Jira,Yes,openAI GPT (chatbot models),openAI GPT (chatbot models),"Yes, I use AI tools weekly","No, and I don't plan to",NaN,104413.0,9.0
3,I am a developer by profession,35-44 years old,44.0,39.0,Bachelor’s degree,"Independent contractor, freelancer, or self-em...",10.0,"Yes, I learned how to use AI-enabled tools for...",12.0,"Developer, front-end",...,GitHub;GitLab;Jira,GitHub;GitLab;Jira,Yes,Gemini (Flash general purpose models);openAI G...,Gemini (Flash general purpose models);Gemini (...,"Yes, I use AI tools daily","Yes, I use AI agents at work weekly",Software engineering,53061.0,8.0
4,I am a developer by profession,35-44 years old,44.0,39.0,Bachelor’s degree,Employed,4.0,"Yes, I learned how to use AI-enabled tools for...",5.0,"Developer, back-end",...,GitLab;Jira;Miro,GitLab;Jira,No,NaN,NaN,"Yes, I use AI tools weekly","Yes, I use AI agents at work monthly or infreq...",Software engineering,36197.0,6.0
5,I am a developer by profession,35-44 years old,44.0,39.0,Master’s degree,"Independent contractor, freelancer, or self-em...",21.0,"Yes, I learned how to use AI-enabled tools for...",22.0,Engineering manager,...,Azure Devops;GitHub;Jira,Azure Devops;GitHub;Jira,Yes,openAI GPT (chatbot models),openAI GPT (chatbot models),"Yes, I use AI tools weekly","No, and I don't plan to",NaN,60000.0,7.0


## Filter die Zeilen raus, die mehr Coding-Erfahrung haben als Lebensalter - 6 Jahre

In [18]:
# Keep only rows where WorkExp is NOT greater than (MaxAge - 16)
df = df[~(df['YearsCode'] > (df['MaxAge'] - 6))]

# Display the first few rows of the filtered dataframe
df.head()

Original number of rows: 47803
Number of rows after filtering: 47723
Removed 80 rows (0.2%)


,MainBranch,Age,MaxAge,AgeNum,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
ResponseId,,,,,,,,,,,,,,,,,,,,,
1,i am a developer by profession,25-34 years old,34.0,29.0,master's degree,employed,"[caring for dependents (children, elderly, etc.)]",8.0,"yes, i am not new to coding but am learning ne...",[online courses or certification (includes all...,...,[vertex ai],[],[],[],[chatgpt],[],[when i don't trust ai's answers],"[troubleshooting, profiling, debugging]",61256.0,10.0
2,i am a developer by profession,25-34 years old,34.0,29.0,associate degree,employed,[],2.0,"yes, i am not new to coding but am learning ne...","[books / physical media, online courses or cer...",...,[],[],[],[],[],[],"[when i don't trust ai's answers, when i have ...",[all skills. ai is a flop.],104413.0,9.0
3,i am a developer by profession,35-44 years old,44.0,39.0,bachelor's degree,"independent contractor, freelancer, or self-em...",[none of the above],10.0,"yes, i am not new to coding but am learning ne...",[online courses or certification (includes all...,...,[],[],[],[],"[chatgpt, claude code, github copilot, google ...",[],"[when i don't trust ai's answers, when i have ...","[understand how things actually work, problem ...",53061.0,8.0
4,i am a developer by profession,35-44 years old,44.0,39.0,bachelor's degree,employed,[none of the above],4.0,"yes, i am not new to coding but am learning ne...","[ai codegen tools or ai-enabled apps, other on...",...,[],[],[],[],"[chatgpt, claude code]",[],"[when i don't trust ai's answers, when i have ...",[],36197.0,6.0
5,i am a developer by profession,35-44 years old,44.0,39.0,master's degree,"independent contractor, freelancer, or self-em...","[caring for dependents (children, elderly, etc.)]",21.0,"no, i am not new to coding and did not learn n...",[],...,[],[],[],[],[],[],[when i don't trust ai's answers],"[critical thinking, the skill to define the ta...",60000.0,7.0


In [19]:
df.to_csv("survey_results_test.csv", index=False)